In [ ]:
# %% [markdown]
# # AI2-THOR rollout: retrieval arms on one interaction clip
# Rolls every arm in RUNS along the evaluation leg of one AI2-THOR v3 test
# episode and compares what each one renders at the station revisits. TOUR
# episodes: interaction chains on the EXPLORATION leg (open / close / pick up /
# put), then a deterministic RETRACE on the EVAL leg whose reveals show the
# changed state. The discriminating case is same-pose-different-state: a
# pose-only memory retrieves the STALE view of a station the agent changed.
# Arms: Temporal / WorldMem (recency / FOV-overlap heuristics) and FAR (the
# learned retriever; FAR -- Multi-Cue gates the visual channel against
# closed-form geometry). Every arm is its own training run with the same
# generator, rolled with the replace-tail protocol (the pool's history stays
# real) and the same noise seed.
#
# Needs: the checkpoint bundles under `models/ai2thor_v3/<arm>/` (each carries
# its `config.yaml` and `checkpoints/<step>.pth.tar`), the pre-computed latents
# under `datasets/ai2thor_v3/latent/`, and the manifest / test indices under
# `results/manifests/ai2thor_v3/` and `results/indices/ai2thor_v3/`. The GPU
# is chosen by the `FAR_DEVICE` environment variable (default `cuda:0`).
# The machinery is `far.analysis.thor_rollout` (one function per cell; see its
# docstrings).

# %%
import os
import numpy as np
import torch
import matplotlib.pyplot as plt

from far.constants import ROOT
from far.analysis import thor_rollout as R

os.chdir(ROOT)
os.environ.setdefault("FAR_MODELS_ROOT", os.path.join(ROOT, "models"))
os.environ.setdefault("FAR_RESULT_ROOT", os.path.join(ROOT, "results"))
device = os.environ.get("FAR_DEVICE", "cuda:0") if torch.cuda.is_available() else "cpu"

RUNS = {   # label -> (run dir, checkpoint step); each run dir carries its own config
    # Keys are the DISPLAY NAMES shown on every figure / video. All three arms
    # were trained on the AI2-THOR v3 corpus with the union loss masks and
    # STATE-conditioned generators (interact_dim=5: interaction counts + door
    # fraction; the rollout feeds action_state); pair them with
    # DATASET_GROUP=ai2thor_v3_latent. Enable several arms to compare them on
    # the same clip.
    "FAR":  ("models/ai2thor_v3/far_multicue", "0700000"),   # FAR -- Multi-Cue (the learned retriever)
    # "Temporal": ("models/ai2thor_v3/temporal", "0700000"),   # recency baseline
    # "WorldMem": ("models/ai2thor_v3/worldmem", "0700000"),   # FOV-overlap baseline
}


In [ ]:
# %%
# Models + RAW test dataset (global poses + cumulative interaction events kept).
POOL_SIZE     = 1000                   # retrieval pool; covers the whole outbound leg
DATASET_GROUP = "ai2thor_v3_latent"
USE_CTRL_SET  = False                  # True = the seed-matched `_ctrl` siblings (same
                                       # CLIP_IDX ordering as the act set). Static twins:
                                       # the same stations are opened/closed (revisit
                                       # windows still work) but nothing is moved — every
                                       # reveal shows an UNCHANGED interior.

models, data_cfg = R.build_models(RUNS, device)
if USE_CTRL_SET:
    R.use_ctrl_index(data_cfg, DATASET_GROUP)
dataset, spec, stats_np, TRAINED_POOL = R.build_dataset(data_cfg, DATASET_GROUP, POOL_SIZE)
decode = R.Decoder(next(iter(models.values())).tokenizer, device)   # shared SDXL VAE


In [ ]:
# %%
# Pick the clip and the rollout window — the only cell to re-run when changing clips.
CLIP_IDX       = 350    # test clip (its position in the test index)
SEED           = 0      # worldmem-subsample seed; also re-seeds rollout noise
ROLLOUT_STRIDE = 1     # GT frames per generation step (fps 10 -> 0.2 s)
SAMPLER, SAMPLING_STEPS = "ddim", 20
PNP_CHUNK_SIZE, CONTEXT_SIZE, WORLDMEM_POOL_SIZE = 10, 12, None   # None = trained value / no cap
POOL_PHASE     = "exploration"   # retrievable history stops at eval_start | "traversed" = walked eval frames join (ablation)

START_OFFSET, MAX_STEPS = 0,None  # the window: frames into the eval leg, then a step cap (None = to the end)
# COUNTERFACTUAL: make an interval of the observation history unretrievable, so the
# arms must answer the reveal without the evidence. An arm that still renders the
# CHANGED state is not reading it from memory. Excluded frames are cleared in the
# pool mask, which every memory honours; the CURRENT frame is never dropped.
EXCLUDE_EVENT    = None            # index, or LIST of indices, into the exploration-leg interaction
                                   #   firings (0 = first, negatives from the end); each is dropped
                                   #   +- EXCLUDE_PAD seconds. e.g. 0 or [0, 2]. None = off.
EXCLUDE_WINDOW   = None            # explicit interval, or LIST of intervals: (a, b) episode FRAMES
                                   #   (ints) or (a, b) SECONDS (floats), e.g. (20.0, 40.0) or
                                   #   [(120, 150), (300.0, 320.0)]. None = off. Unioned with
                                   #   EXCLUDE_EVENT if both are set.
EXCLUDE_PAD      = 1.0             # seconds of padding each side of EXCLUDE_EVENT
SAVE_DIR = f"results/rollout_video_ai2thor/{CLIP_IDX}"; os.makedirs(SAVE_DIR, exist_ok=True)

R.install_eval_knobs(models, data_cfg, SAMPLER, SAMPLING_STEPS, PNP_CHUNK_SIZE, CONTEXT_SIZE)
ep = R.load_episode(dataset, CLIP_IDX, SEED)
revisits = R.print_revisits(ep, spec)
targets = R.apply_window(ep, spec, START_OFFSET, POOL_PHASE, ROLLOUT_STRIDE, MAX_STEPS)
R.exclude_from_pool(ep, spec, window=EXCLUDE_WINDOW, event=EXCLUDE_EVENT, pad_s=EXCLUDE_PAD)
R.inspect_trajectory(ep, spec, decode)


In [ ]:
# %%
# Full-episode video: the clip from its first frame to its last, GROUND TRUTH ONLY, in the rollout video's
# style — one tight panel: "History / evaluation in N s" (action captions, agent sightings), the transition
# card, then "Ground truth / evaluation" (expected-state captions, boxes, reveal holds). Every frame comes
# from the episode's latents.npy at ROLLOUT_STRIDE spacing (~1-2 min of decoding).
HISTORY_SPEED, EVAL_SPEED = 2, 2   # playback speed of each phase, x real time (integers; encoded at their product x the frame rate)
R.write_episode_video(ep, spec, decode, f"{SAVE_DIR}/full_episode.mp4", history_speed=HISTORY_SPEED, eval_speed=EVAL_SPEED, outer_pad=0)


In [ ]:
# %%
# Roll every arm along the same window (same noise seed); record each arm's picks.
preds, pick_times, pick_scores, step_interact = R.run_rollouts(
    models, ep, spec, stats_np, device, sampler=SAMPLER, seed=SEED,
    worldmem_pool_size=WORLDMEM_POOL_SIZE)
gt_lat   = ep["goal_lat"][targets]
gt_img   = decode(gt_lat)
cur_img  = decode(ep["ctx_lat"][-1:])[0]
pred_img = {l: decode(p) for l, p in preds.items()}


In [ ]:
# %%
# Per-step metric + comparison grid. Whole-frame metrics get outvoted by
# fidelity-everywhere-else when the error is one small object; the station /
# center variants target the object (see R.metric_curves).
METRIC = "lpips"   # psnr | lpips | dreamsim | latent_top32 | station | station_latent | center_lpips | center_dreamsim

mvals, M_FMT, M_UNIT, M_BETTER = R.metric_curves(METRIC, preds, pred_img, gt_lat, gt_img, ep, device, revisits)
R.plot_compare(ep, spec, cur_img, gt_img, pred_img, mvals, step_interact, M_FMT, M_UNIT,
               title=f"{ep['video_id']} — rollout @ stride {ROLLOUT_STRIDE} ({SAMPLER}{SAMPLING_STEPS})",
               save_path=f"{SAVE_DIR}/compare.png")
R.plot_metric_curve(ep, spec, mvals, step_interact, METRIC, M_FMT, M_UNIT, M_BETTER, f"{SAVE_DIR}/{METRIC}.png")


In [ ]:
# %%
# Rollout video: GT vs each arm per step, with the context frames it retrieved ("Context #k from -t sec", best-ranked first).
PICK_TOP_N = 4      # picks shown per arm per step (None = all)
SAVE_FRAME = 4   # step index (int or list, negative OK) -> grid also saved as rollout_step<k>.png; None = video only
INCLUDE_INTERACTION_PHASE = True   # True = play the episode from its start with GT ONLY (arms blank) until the GT reaches the
                                    # evaluation phase, then the rollouts start; the GT tag names the phase (history / evaluation)
DRAW_CHANGE_BOX = True              # boxes around the GT target's CHANGE REGIONS (latents.change.npy) on the GT and arm tiles (white;
                                    # green / red on an arm tile = its VERDICT below),
                                    # only while the GT is AT a station (revisit arrival -> leaving; not the walk between): one box per
                                    # connected region of >= 4 tokens (specks ignored); skipped when the frame's mask is empty,
                                    # exceeds the loss gate (10% of the grid) or the boxes would cover half the frame

ACTION_LABEL_S     = 2.0           # green caption on the GT tile for this long after each interaction ("Pick up apple"); 0 = off
STATE_LABEL        = True          # coral expected-state caption at CHANGED-station revisits ("Apple should be here"); observed-only stations get none
VERDICT            = True          # per-arm Correct/Wrong state at those revisits, from LPIPS inside the change box: a step is CORRECT when
VERDICT_LPIPS_MAX  = 0.30         #   d_a = LPIPS(pred, GT) is below this ABSOLUTE ceiling (a render far from everything fails) AND
VERDICT_MARGIN     = 0.02         #   d_b - d_a (closer to the after-state than to the exact-view BEFORE frame) exceeds this, unless
VERDICT_SEP_MIN    = 0.05         #   sep = LPIPS(GT, before) is under this — the two states are then indistinguishable, so only the ceiling applies.
VERDICT_BINARY     = True         #   Verdict = majority over the revisit's steps; True = no unlabelled arms (anything not correct is Wrong state).
                                  #   The per-revisit d_a / d_b / sep means are printed — calibrate the ceiling from them.
REVEAL_HOLD_S      = 2.0           # freeze the middle step of each changed-station revisit for this long (inspection time); 0 = off
PHASE_CARD_S       = 1.5           # "Evaluation phase" transition card at eval_start (with INCLUDE_INTERACTION_PHASE); 0 = off
GUTTER, BOTTOM_PAD = 10, 64        # px between tiles / canvas strip under the grid (keeps a player's control bar off the tiles)
LAYOUT = "panels"                  # "panels" = every row in a rounded panel with a name card (verdict pill, coloured edge); "grid" = plain tiles
OURS   = "FAR"                     # the arm drawn as ours in the name card (amber, "(ours)"); a RUNS label, None = none

R.write_rollout_video(ep, spec, decode, gt_img, pred_img, mvals, M_FMT, M_UNIT, step_interact, pick_times, pick_scores,
                      f"{SAVE_DIR}/rollout.mp4", pick_top_n=PICK_TOP_N, save_frame=SAVE_FRAME,
                      include_interaction_phase=INCLUDE_INTERACTION_PHASE, change_box=DRAW_CHANGE_BOX,
                      action_label_seconds=ACTION_LABEL_S, state_label=STATE_LABEL, verdict=VERDICT, verdict_margin=VERDICT_MARGIN, verdict_lpips_max=VERDICT_LPIPS_MAX,
                      verdict_sep_min=VERDICT_SEP_MIN, verdict_binary=VERDICT_BINARY,
                      reveal_hold_seconds=REVEAL_HOLD_S, phase_card_seconds=PHASE_CARD_S, gutter=GUTTER, bottom_pad=BOTTOM_PAD, device=device,
                      layout=LAYOUT, ours=OURS)


In [ ]:
# %%
# Change-region diagnostic: error INSIDE the GT change mask, and the copy
# signature (is the prediction closer to the retrieved frame than to the truth?).
CHG_MAX_FRAC = 0.10   # skip masks above this share of the grid (the loss's gate; raise to ~0.2 for large items like towels)

chg_rows = R.change_region_diagnostic(ep, spec, preds, pred_img, gt_lat, gt_img, pick_times, pick_scores, device,
                                      max_frac=CHG_MAX_FRAC, save_path=f"{SAVE_DIR}/change_region.png")

In [ ]:
# %%
# UPSCALED rollout video: the rollout video above, reproduced with the ROLLOUT frames (GT target + each arm's
# generation) passed through SwinIR (far.analysis.upscale). The context thumbnails, history prelude and
# verdict before-views stay native decodes,
# bicubic-fitted to the upscaled tile size — only the frames the arms produced carry the super-resolution.
# Deterministic, the same chain for GT and every arm; metrics / labels / verdicts are computed on the
# frames as shown. Demo material only.
import far.analysis.upscale as U
SR_MODEL = "classical x4"                    # "classical x2" = most faithful | "real GAN x2" / "real-L GAN x4" = sharpest
sr_net, SR_SCALE = U.load_swinir(SR_MODEL, device)   # weights auto-download to models/swinir/ once
from tqdm.auto import tqdm
with tqdm(total=gt_img.shape[0] * (1 + len(pred_img)), desc=f"SwinIR {SR_MODEL}", unit="frame") as sr_bar:   # one bar: GT + every arm
    def _sr(v, src): sr_bar.set_postfix(src=src); return U.sr_tensor(sr_net, v, SR_SCALE, pbar=sr_bar)
    gt_up   = _sr(gt_img, "GT")
    pred_up = {l: _sr(v, l) for l, v in pred_img.items()}
R.write_rollout_video(ep, spec, decode, gt_up, pred_up, mvals, M_FMT, M_UNIT, step_interact, pick_times, pick_scores,
                      f"{SAVE_DIR}/rollout_x{SR_SCALE}.mp4", pick_top_n=PICK_TOP_N, save_frame=SAVE_FRAME,
                      include_interaction_phase=INCLUDE_INTERACTION_PHASE, change_box=DRAW_CHANGE_BOX,
                      action_label_seconds=ACTION_LABEL_S, state_label=STATE_LABEL, verdict=VERDICT, verdict_margin=VERDICT_MARGIN, verdict_lpips_max=VERDICT_LPIPS_MAX,
                      verdict_sep_min=VERDICT_SEP_MIN, verdict_binary=VERDICT_BINARY,
                      reveal_hold_seconds=REVEAL_HOLD_S, phase_card_seconds=PHASE_CARD_S, gutter=GUTTER * SR_SCALE,
                      bottom_pad=BOTTOM_PAD * SR_SCALE, device=device, layout=LAYOUT, ours=OURS)


In [ ]:
# %%
# Retrieval geography: which TIME each arm reads from, per step (shaded = exploration phase).
R.plot_retrieval_geography(ep, spec, pick_times, pick_scores, step_interact, f"{SAVE_DIR}/retrieval_geography.png")


In [ ]:
# %%
# Save every frame as a PNG (re-layout / inspection outside the notebook): the
# current frame, the GROUND-TRUTH frames (the whole clip by default), the GT at
# each rollout target, each arm's generated frame per step, and the TOP-1 context
# frame that arm RETRIEVED for that step ->
# SAVE_DIR/frames/{gt,gt_targets,<arm>,<arm>_top1}/.
# File names carry the episode frame index (f<idx>, the meta.json axis) and the
# rollout step (step<k>), so GT targets, generations and picks pair up by name;
# a top-1 name also carries the retrieved frame itself (_ctx<fi>).
GT_SCOPE   = None   # ground-truth frames to dump, each named f<idx>_explore/_eval:
                         #   "episode" = the SAMPLED clip (pool + trajectory), the span the
                         #     full-episode video covers (~1-2 min of decoding). Under
                         #     START_OFFSET > 0 with POOL_PHASE "exploration" the eval frames
                         #     WALKED BEFORE the window are absent — they join neither the pool
                         #     nor the sliced trajectory;
                         #   "clip"    = every frame of latents.npy, window-independent (use it
                         #     when the rollout starts mid-trajectory and you still want the walk
                         #     up to it);
                         #   "eval"    = the eval leg only; None = skip.
CTX_FRAMES = False       # also dump EVERY retrievable pool frame (POOL_SIZE files); the per-step
                         #   top-1 picks above are written regardless

R.export_frames(ep, targets, decode, gt_img, pred_img, cur_img, SAVE_DIR,
                scope=GT_SCOPE, ctx_frames=CTX_FRAMES, pick_times=pick_times, spec=spec)
# ...and the numbers that go with them -> SAVE_DIR/meta.json: per step, per arm, the metric,
# the retrieved pick times AND those picks as episode frame indices, so the frames behind the
# WHOLE retrieved set can be read out of frames/gt/f<idx>_explore.png (not just the top-1 copy).
R.export_records(ep, spec, targets, pred_img, pick_times, SAVE_DIR, metrics={METRIC: mvals},
                 step_interact=step_interact, clip_idx=CLIP_IDX, seed=SEED, sampler=SAMPLER,
                 sampling_steps=SAMPLING_STEPS, pool_size=POOL_SIZE, pool_phase=POOL_PHASE,
                 context_size=CONTEXT_SIZE, pnp_chunk_size=PNP_CHUNK_SIZE,
                 runs={l: list(RUNS[l]) for l in pred_img if l in RUNS})
